# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. `[1. 2. 3.] [0 1 1]` — оператор `/` выполняет обычное деление с плавающей точкой (`float`), а `//` — целочисленное деление с отбрасыванием дробной части (`2 // 4 = 0`, `4 // 4 = 1`, `6 // 4 = 1`).
2. `[4. 4. 4.]` — матрица имеет размер 3 строки на 4 столбца и заполнена единицами; параметр `axis=1` суммирует элементы вдоль столбцов (построчно), поэтому в каждой из 3 строк сумма единиц равна 4.
3. `[[0 1 2], [1 2 3], [2 3 4]]` — срабатывает механизм broadcasting: одномерный массив формы `(3,)` и столбец формы `(3, 1)` расширяются до матрицы `3×3`, где каждый элемент равен сумме индексов строки и столбца.
4. `6.0` — `np.sqrt(x)` поэлементно извлекает квадратный корень, получая `[1. 2. 3.]`, а `.sum()` складывает их (`1 + 2 + 3 = 6.0`).
5. `7.5` — `np.arange(1, 5)` создает массив `[1, 2, 3, 4]`, возведение в квадрат дает `[1, 4, 9, 16]`, а среднее арифметическое этих чисел равно `(1 + 4 + 9 + 16) / 4 = 30 / 4 = 7.5`.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

In [ ]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [ ]:
# 1
clean_count = clean.sum()
rev_sep = amount[clean & (month == "2026-09")].sum()
rev_oct = amount[clean & (month == "2026-10")].sum()
pct_change = (rev_oct - rev_sep) / rev_sep * 100

print(f"Строк в clean: {clean_count}")
print(f"Выручка сентябрь: {rev_sep:,.2f} ₸ | октябрь: {rev_oct:,.2f} ₸ | изменение: {pct_change:+.2f}%")
print(f"Менеджеру: После очистки от дублей, тестов и неоплаченных заказов осталось {clean_count} покупок, а выручка в октябре изменилась на {pct_change:+.1f}% по сравнению с сентябрем (цифры полностью совпадают с расчетами из ДЗ 2).")

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [ ]:
# 2
categories = np.unique(cat)
months = ["2026-09", "2026-10"]

cat_rev_table = np.zeros((len(categories), len(months)))
for i, c in enumerate(categories):
    for j, m in enumerate(months):
        cat_rev_table[i, j] = amount[clean & (cat == c) & (month == m)].sum()

total_by_cat = cat_rev_table.sum(axis=1)
top_cat_idx = total_by_cat.argmax()
top_cat = categories[top_cat_idx]
top_cat_rev = total_by_cat[top_cat_idx]

cat_pct_change = (cat_rev_table[:, 1] - cat_rev_table[:, 0]) / cat_rev_table[:, 0] * 100
worst_cat_idx = cat_pct_change.argmin()
worst_cat = categories[worst_cat_idx]
worst_cat_drop = cat_pct_change[worst_cat_idx]

print("Таблица выручки (категория × месяц):\n", cat_rev_table)
print(f"Лидер по выручке: {top_cat} ({top_cat_rev:,.2f} ₸)")
print(f"Сильнее всех упала: {worst_cat} ({worst_cat_drop:+.2f}%)")
print(f"Менеджеру: Главный драйвер выручки за два месяца — категория «{top_cat}» ({top_cat_rev:,.0f} ₸), а сильнее всего в октябре просела категория «{worst_cat}» ({worst_cat_drop:.1f}%).")

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [ ]:
# 3
cat_cnt_table = np.zeros((len(categories), len(months)))
for i, c in enumerate(categories):
    for j, m in enumerate(months):
        cat_cnt_table[i, j] = (clean & (cat == c) & (month == m)).sum()

avg_check_table = cat_rev_table / cat_cnt_table
oct_avg_checks = avg_check_table[:, 1]
max_check_idx = oct_avg_checks.argmax()
max_check_cat = categories[max_check_idx]
max_check_val = oct_avg_checks[max_check_idx]

print("Средний чек в октябре по категориям:", dict(zip(categories, np.round(oct_avg_checks, 2))))
print(f"Максимальный средний чек в октябре: {max_check_cat} — {max_check_val:,.2f} ₸")
print(f"Менеджеру: Самый высокий средний чек в октябре зафиксирован в категории «{max_check_cat}» и составил {max_check_val:,.0f} ₸.")

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4
cities = np.unique(city)
table = np.zeros((len(cities), len(months)))
for i, c in enumerate(cities):
    for j, m in enumerate(months):
        table[i, j] = amount[clean & (city == c) & (month == m)].sum()

city_shares = table / table.sum(axis=0)
print("Сумма долей по столбцам (должно быть [1. 1.]):", city_shares.sum(axis=0))

shym_idx = np.where(cities == "Шымкент")[0][0]
shym_sep, shym_oct = city_shares[shym_idx, 0], city_shares[shym_idx, 1]
shym_diff_pp = (shym_oct - shym_sep) * 100

print(f"Доля Шымкента: сентябрь = {shym_sep:.2%}, октябрь = {shym_oct:.2%} (изменение: {shym_diff_pp:+.2f} п.п.)")
print(f"Менеджеру: Доля Шымкента в общей выручке компании изменилась с {shym_sep:.1%} в сентябре до {shym_oct:.1%} в октябре ({shym_diff_pp:+.1f} процентных пункта).")

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [ ]:
# 5
city_diff = table[:, 1] - table[:, 0]
total_diff = city_diff.sum()
drop_contribution = (city_diff / total_diff) * 100

shym_contrib = drop_contribution[shym_idx]
print("Вклад городов в общее изменение выручки (%):", dict(zip(cities, np.round(drop_contribution, 2))))
print(f"Вклад Шымкента: {shym_contrib:.2f}%")
print(f"Менеджеру: Шымкент обеспечил {shym_contrib:.1f}% от всего падения выручки — причину стоит искать в локальной логистике, работе склада или оттоке клиентов именно в этом регионе.")

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6
# Объяснение:
# Массив `table` имеет форму (6, 2) (6 городов, 2 месяца).
# Выражение `table.sum(axis=1)` возвращает одномерный массив формы (6,).
# По правилам broadcasting NumPy выравнивает размерности с правого края: (6, 2) и (6,) -> (6, 2) и (1, 6).
# Так как 2 != 6, возникает ошибка ValueError: operands could not be broadcast together.
# Чтобы делить построчно, нужно сохранить размерность (6, 1) с помощью `keepdims=True` (или `.reshape(-1, 1)`).

share = table / table.sum(axis=1, keepdims=True)
print("Исправленные доли внутри каждого города (сентябрь / октябрь):\n", np.round(share, 4))
print("Менеджеру: Исправленный расчет корректно показывает распределение выручки между сентябрем и октябрем внутри каждого отдельного города.")

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()

outliers_mask = np.abs(z) > 3
outliers_count = outliers_mask.sum()
outliers_share = outliers_mask.mean()

print(f"Проверка z: mean = {z.mean():.2e}, std = {z.std():.6f}")
print(f"Заказов с |z| > 3: {outliers_count} ({outliers_share:.2%})")
print(f"Менеджеру: В чистых продажах обнаружено {outliers_count} аномально крупных заказов ({outliers_share:.2%} от общего числа), чья сумма отклоняется от среднего более чем на три стандартных отклонения.")

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8
valid_ml_mask = clean & ~np.isnan(days)
X = np.column_stack([amount[valid_ml_mask], days[valid_ml_mask]])

X_scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

print("Проверка min по столбцам:", X_scaled.min(axis=0))
print("Проверка max по столбцам:", X_scaled.max(axis=0))
print("Зачем нужен единый масштаб: Без масштабирования признак с большими числами (сумма заказа в десятках тысяч тенге) будет математически подавлять признак с маленькими числами (дни доставки от 1 до 10) при расчете расстояний или градиентов в ML-моделях.")
print(f"Менеджеру: Подготовлена и нормализована матрица из {len(X_scaled)} заказов, где сумма покупки и срок доставки приведены к единой шкале от 0 до 1 для корректного обучения алгоритмов прогнозирования.")